# PKG — Ensured Subrogation Network

**Purpose:** produce ONE bounded, defensible, visualisable subrogation
network from the monthly PKG snapshots. This is deliberately **not** the
exploratory notebook: every stage here is a *gate* that discards cases, and
the output is capped at a few hundred nodes so it can actually be shown to
a stakeholder.

**Design stance — precision over recall.** There is no transaction-level
ground truth for subrogation today (no confirmed labels from Arbitration
Forums, claims systems, or ACH addenda). So "ensured" here means
**highest structural confidence with ambiguous cases deliberately
discarded** — it does *not* mean verified. Every case that survives should
be individually defensible; anything needing a caveat is dropped rather
than tiered down.

**Funnel:**

| Stage | Gate |
|---|---|
| 1–4 | Load, parse NAICS, aggregate window, measure persistence |
| 5 | **Entity gate** — NAICS-verified insurers / law firms only |
| 6 | **Carrier↔carrier gate** — reciprocal + persistent + balanced + not reinsurance |
| 7 | **Carrier↔law-firm gate** — recovery-direction + persistent |
| 8 | **Clearinghouse labelling** |
| 9 | **Node budget** — greedy top-scoring pairs until the cap |
| 10 | Visualise |
| 11 | Persist + run manifest |

Every gate prints what it dropped. That drop log *is* the audit trail.

In [ ]:
import json
import math
import time
from pathlib import Path
from typing import Dict, List, Optional, Set

import numpy as np
import pandas as pd

## 0. Config — every threshold in one place

Thresholds are strict by default. They are **structural heuristics, not
calibrated cutoffs**: tune them against your real distributions (the
notebook prints the relevant distribution before each gate that uses one).

In [ ]:
DATA_DIR = Path("../data")
FILE_PREFIX = "cust_"
START_YM = "2025-01"
END_YM = "2025-11"

OUT_DIR = Path("./ensured_output")

# --- Entity gate -------------------------------------------------------
# Strict mode requires a VALID NAICS to admit an entity. Name patterns alone
# are not enough for the "ensured" set -- they were the source of the
# accounting-LLP style false positives. Trust-account wording is the one
# name signal allowed to stand alone (IOLTA / "client trust account" is
# unambiguous and frequently the only marker when NAICS is missing).
STRICT_REQUIRE_VALID_NAICS = True
INSURER_NAICS3 = ["524"]
LAW_FIRM_NAICS4 = ["5411"]

INSURER_NAME_PATTERN = r"\b(?:insurance|assurance|casualty|indemnity|reinsurance|underwriters?)\b"
LAW_FIRM_NAME_PATTERN = (
    r"\b(?:law\s+offices?|law\s+firm|law\s+group|law\s+center|attorneys?|"
    r"legal\s+services|legal\s+group|counsell?ors?\s+at\s+law|"
    r"esq(?:uire)?|barristers?|solicitors?)\b"
)
TRUST_ACCOUNT_PATTERN = (
    r"\b(?:iolta|client\s+trust|attorney\s+trust|lawyers?\s+trust|"
    r"trust\s+account|client\s+funds)\b"
)

# --- Persistence gate --------------------------------------------------
# Fraction of the window a relationship must recur in, and the longest
# unbroken run it must show. Both, not either: scattered months and a short
# burst are different failure modes.
MIN_MONTH_FRACTION = 0.55
MIN_STREAK_FRACTION = 0.35

# --- Carrier<->carrier gate --------------------------------------------
# balance_ratio = min(amt_fwd, amt_rev) / max(...). A token reverse payment
# is not reciprocity; require real two-way movement.
MIN_BALANCE_RATIO = 0.30
# Reinsurance treaty flows look structurally identical to subrogation
# (carrier<->carrier, reciprocal) but are FEW and LARGE. Cut on a percentile
# of the observed avg-amount-per-txn distribution rather than an absolute
# dollar figure, so it travels across windows and books.
REINSURANCE_AVG_AMT_PCTL = 0.90

# --- Carrier<->law-firm gate -------------------------------------------
# recovery_ratio = share of pair money flowing FIRM -> INSURER.
# >= HIGH  : recovery inflow (lien / subrogation settlement)  -> KEEP
# <= LOW   : outbound defense panel / litigation funding      -> DROP
# between  : two-way recovery counsel (net of contingency fee)-> KEEP
RECOVERY_RATIO_HIGH = 0.80
RECOVERY_RATIO_LOW = 0.20

# --- Clearinghouse labelling -------------------------------------------
# A node reciprocally connected to at least this share of admitted insurers
# behaves like an arbitration/settlement clearinghouse (the Arbitration
# Forums pattern) rather than a normal counterparty.
CLEARINGHOUSE_INSURER_FRACTION = 0.50

# --- Node budget -------------------------------------------------------
# Hard cap on the final network. Pairs are added best-first until adding the
# next one would exceed this.
MAX_NODES = 200
# Cap edges drawn per clearinghouse hub so one node can't swamp the layout.
MAX_HUB_EDGES_DRAWN = 40

OUT_DIR.mkdir(parents=True, exist_ok=True)


def month_range(start_ym: str, end_ym: str) -> List[str]:
    return pd.date_range(pd.Timestamp(start_ym + "-01"),
                         pd.Timestamp(end_ym + "-01"), freq="MS").strftime("%Y-%m").tolist()


MONTHS = month_range(START_YM, END_YM)
N_WINDOW = len(MONTHS)
MIN_MONTHS = max(2, int(np.ceil(MIN_MONTH_FRACTION * N_WINDOW)))
MIN_STREAK = max(2, int(np.ceil(MIN_STREAK_FRACTION * N_WINDOW)))

print(f"Window: {MONTHS[0]} .. {MONTHS[-1]}  ({N_WINDOW} months)")
print(f"Persistence gate: >= {MIN_MONTHS} active months AND >= {MIN_STREAK} consecutive")
print(f"Node budget: {MAX_NODES}")

# Drop log -- every gate appends to this; printed and persisted at the end.
DROP_LOG: List[dict] = []


def log_gate(stage: str, kept: int, dropped: int, note: str = "") -> None:
    DROP_LOG.append({"stage": stage, "kept": kept, "dropped": dropped, "note": note})
    total = kept + dropped
    pct = (100.0 * kept / total) if total else 0.0
    print(f"  [GATE] {stage:<38} kept {kept:>7,} / {total:>7,} ({pct:5.1f}%)"
          + (f"  -- {note}" if note else ""))

## 1. Load the window

In [ ]:
RAW_DTYPES = {
    "source": "string", "source_name": "string", "source_naics": "string",
    "amount": "float64", "volume": "float64",
    "dest": "string", "dest_name": "string", "dest_naics": "string",
}

frames, missing = [], []
t0 = time.time()
for ym in MONTHS:
    p = DATA_DIR / f"{FILE_PREFIX}{ym}.csv"
    if not p.exists():
        missing.append(str(p))
        continue
    df = pd.read_csv(p, dtype=RAW_DTYPES)
    df["year_month"] = ym
    frames.append(df)

if missing:
    print(f"WARNING: {len(missing)} snapshot(s) missing and skipped:")
    for m in missing[:5]:
        print(f"  - {m}")
if not frames:
    raise FileNotFoundError(
        f"No snapshots found in {DATA_DIR.resolve()} for {MONTHS[0]}..{MONTHS[-1]}. "
        f"Check DATA_DIR / FILE_PREFIX.")

raw = pd.concat(frames, ignore_index=True)
del frames
print(f"Loaded {len(raw):,} rows from {N_WINDOW - len(missing)} snapshots "
      f"in {time.time() - t0:.1f}s ({raw.memory_usage(deep=True).sum() / 1e9:.2f} GB)")

## 2. NAICS parsing

Splits `CODE|DESCRIPTION` and derives 2–6 digit rollups. Known dirty
sentinels are marked invalid rather than truncated into a fake code —
which matters here, because the entity gate keys off `naics_valid`.
`naics_observed` is never conflated with any imputed value.

In [ ]:
NAICS_SENTINELS = {"-1", "", "UNKNOWN", "******"}


def split_naics(s_raw: pd.Series) -> pd.DataFrame:
    s = s_raw.fillna("").astype(str)
    parts = s.str.split("|", n=1, expand=True)
    code = parts[0].str.strip()
    desc = parts[1].str.strip() if parts.shape[1] > 1 else pd.Series("", index=s.index)
    valid = code.str.fullmatch(r"\d{2,6}").fillna(False) & ~code.isin(NAICS_SENTINELS)
    out = pd.DataFrame({"naics_code": code.where(valid),
                        "naics_desc": desc.where(valid),
                        "naics_valid": valid})
    for n in (2, 3, 4, 5, 6):
        out[f"naics{n}"] = out["naics_code"].str.slice(0, n).where(
            (out["naics_code"].str.len() >= n).fillna(False))
    return out


raw = pd.concat([raw.drop(columns=["source_naics", "dest_naics"]),
                 split_naics(raw["source_naics"]).add_prefix("source_"),
                 split_naics(raw["dest_naics"]).add_prefix("dest_")], axis=1)
print(f"NAICS valid -- source {raw['source_naics_valid'].mean():.1%}, "
      f"dest {raw['dest_naics_valid'].mean():.1%}")

## 3. Node table

In [ ]:
_NCOLS = ["naics_code", "naics_desc", "naics_valid",
          "naics2", "naics3", "naics4", "naics5", "naics6"]

_src = raw[["source", "source_name", "year_month"] + [f"source_{c}" for c in _NCOLS]].copy()
_src.columns = ["node_id", "name", "year_month"] + _NCOLS
_dst = raw[["dest", "dest_name", "year_month"] + [f"dest_{c}" for c in _NCOLS]].copy()
_dst.columns = ["node_id", "name", "year_month"] + _NCOLS

nodes = (pd.concat([_src, _dst], ignore_index=True)
         .sort_values("year_month")
         .groupby("node_id", sort=False).last()      # last non-null per attribute
         .drop(columns="year_month").reset_index())
del _src, _dst
print(f"Unique nodes: {len(nodes):,}")

## 4. Aggregate the window + persistence shape

`n_months_active` counts DISTINCT months the pair transacted (the stated
"one rel per snapshot" definition). `max_streak` is the longest unbroken
run — a steady relationship vs. scattered one-offs.

In [ ]:
edges = (raw.groupby(["source", "dest"], sort=False)
         .agg(amount_total=("amount", "sum"),
              volume_total=("volume", "sum"),
              n_months_active=("year_month", "nunique"))
         .reset_index())
edges["avg_amount_per_txn"] = edges["amount_total"] / edges["volume_total"].replace(0, np.nan)

# sanity: the "one row per pair per snapshot" assumption
_dupe = raw.groupby(["source", "dest", "year_month"], sort=False).size()
_n_dupe = int((_dupe > 1).sum())
if _n_dupe:
    print(f"NOTE: {_n_dupe:,} pair-months have >1 row (summed). n_months_active "
          f"still counts distinct months, so it is unaffected.")

edge_month = (raw.groupby(["source", "dest", "year_month"], sort=False)
              .agg(amount=("amount", "sum")).reset_index())
edge_month["month_idx"] = edge_month["year_month"].map({ym: i for i, ym in enumerate(MONTHS)})
edge_month = edge_month.sort_values(["source", "dest", "month_idx"], kind="mergesort")

# longest consecutive run, vectorised (a gap != 1 opens a new run; the first
# row of each pair yields NaN which also compares != 1)
_gap = edge_month.groupby(["source", "dest"], sort=False)["month_idx"].diff()
edge_month["run_id"] = _gap.ne(1).cumsum()
_streak = (edge_month.groupby(["source", "dest", "run_id"], sort=False).size()
           .reset_index(name="run_len")
           .groupby(["source", "dest"], sort=False)["run_len"].max()
           .rename("max_streak").reset_index())

edges = edges.merge(_streak, on=["source", "dest"], how="left")
edges["max_streak"] = edges["max_streak"].fillna(1).astype(int)
print(f"Directed edges: {len(edges):,}   "
      f"(median months active {edges['n_months_active'].median():.0f}, "
      f"median streak {edges['max_streak'].median():.0f})")

## 5. GATE — entity admission

Only NAICS-verified insurers and law firms are admitted. Name-only matches
are **rejected** in strict mode: they were the source of false positives
(an accounting firm sharing the "LLP" suffix, a medical practice sharing
"PC"). The single exception is trust-account wording, which is admitted on
its own because IOLTA / "client trust account" is unambiguous and is
frequently the *only* signal when NAICS is missing — exactly the WC/GL
settlement channel we most want to keep.

In [ ]:
_name = nodes["name"].fillna("")
_valid = nodes["naics_valid"].fillna(False)

_ins_naics = nodes["naics3"].fillna("").isin(INSURER_NAICS3) & _valid
_ins_name = _name.str.contains(INSURER_NAME_PATTERN, case=False, regex=True, na=False)
_law_naics = nodes["naics4"].fillna("").isin(LAW_FIRM_NAICS4) & _valid
_law_name = _name.str.contains(LAW_FIRM_NAME_PATTERN, case=False, regex=True, na=False)
_trust = _name.str.contains(TRUST_ACCOUNT_PATTERN, case=False, regex=True, na=False)

if STRICT_REQUIRE_VALID_NAICS:
    is_insurer = _ins_naics
    is_law_firm = (_law_naics | _trust) & ~is_insurer
else:
    is_insurer = _ins_naics | _ins_name
    is_law_firm = (_law_naics | _law_name | _trust) & ~is_insurer

nodes["node_class"] = np.select([is_insurer, is_law_firm], ["insurer", "law_firm"],
                                default="other")
nodes["has_trust_flag"] = _trust & ~is_insurer
# what strict mode threw away -- surface it so the cost of precision is visible
nodes["rejected_name_only_insurer"] = _ins_name & ~_ins_naics & (nodes["node_class"] == "other")
nodes["rejected_name_only_law"] = _law_name & ~_law_naics & ~_trust & (nodes["node_class"] == "other")

n_ins = int((nodes["node_class"] == "insurer").sum())
n_law = int((nodes["node_class"] == "law_firm").sum())
log_gate("entity: admitted (insurer|law_firm)", n_ins + n_law, len(nodes) - n_ins - n_law,
         f"{n_ins} insurers, {n_law} law firms")
print(f"  insurers admitted : {n_ins:,}")
print(f"  law firms admitted: {n_law:,}  (trust-flagged {int(nodes['has_trust_flag'].sum()):,})")
print(f"  rejected, name-only insurer-like: {int(nodes['rejected_name_only_insurer'].sum()):,}")
print(f"  rejected, name-only law-like    : {int(nodes['rejected_name_only_law'].sum()):,}")
if n_ins == 0:
    raise ValueError("No insurers admitted -- check INSURER_NAICS3 against your data "
                     "(see the NAICS EDA in the exploratory notebook), or set "
                     "STRICT_REQUIRE_VALID_NAICS = False.")

cls_of = nodes.set_index("node_id")["node_class"]
name_of = nodes.set_index("node_id")["name"]
trust_of = nodes.set_index("node_id")["has_trust_flag"]

edges["source_class"] = edges["source"].map(cls_of)
edges["dest_class"] = edges["dest"].map(cls_of)

## 6. GATE — carrier ↔ carrier ensured pairs

Four conditions, all required:

1. **Reciprocal** — edges in both directions across the window. Checked on
   the aggregate, never per-month: recovery lags the original claim
   payment by weeks to months, so same-month reciprocity would badly
   undercount real relationships.
2. **Persistent** — meets both the active-months and consecutive-streak gates.
3. **Balanced** — a token reverse payment is not a two-way relationship.
4. **Not reinsurance-shaped** — treaty settlement is also reciprocal
   carrier↔carrier, but few-and-large. Cut on a percentile of the observed
   avg-amount distribution.

In [ ]:
ii = edges.loc[(edges["source_class"] == "insurer") & (edges["dest_class"] == "insurer")].copy()
print(f"Insurer->insurer directed edges: {len(ii):,}")

_pair_cols = ["amount_total", "volume_total", "n_months_active", "max_streak", "avg_amount_per_txn"]
_m = ii.merge(ii, left_on=["source", "dest"], right_on=["dest", "source"], suffixes=("_f", "_r"))
_m = _m.loc[_m["source_f"] < _m["dest_f"]]          # one row per unordered pair
cc = _m.rename(columns={"source_f": "a", "dest_f": "b"})[
    ["a", "b"] + [f"{c}_{s}" for c in _pair_cols for s in ("f", "r")]].copy()
n_recip = len(cc)
log_gate("cc: reciprocal (both directions)", n_recip, max(0, len(ii) - n_recip * 2),
         "aggregate window, not per-month")

if len(cc):
    cc["months_min"] = cc[["n_months_active_f", "n_months_active_r"]].min(axis=1)
    cc["streak_max"] = cc[["max_streak_f", "max_streak_r"]].max(axis=1)
    cc["amount_both"] = cc["amount_total_f"] + cc["amount_total_r"]
    cc["volume_both"] = cc["volume_total_f"] + cc["volume_total_r"]
    cc["balance_ratio"] = (cc[["amount_total_f", "amount_total_r"]].min(axis=1) /
                           cc[["amount_total_f", "amount_total_r"]].max(axis=1))
    cc["avg_amt_pair"] = cc["amount_both"] / cc["volume_both"].replace(0, np.nan)

    _before = len(cc)
    cc = cc.loc[(cc["months_min"] >= MIN_MONTHS) & (cc["streak_max"] >= MIN_STREAK)]
    log_gate("cc: persistence", len(cc), _before - len(cc),
             f">= {MIN_MONTHS} months AND >= {MIN_STREAK} streak")

    _before = len(cc)
    cc = cc.loc[cc["balance_ratio"] >= MIN_BALANCE_RATIO]
    log_gate("cc: balanced reciprocity", len(cc), _before - len(cc),
             f"balance_ratio >= {MIN_BALANCE_RATIO}")

    if len(cc):
        reins_cut = cc["avg_amt_pair"].quantile(REINSURANCE_AVG_AMT_PCTL)
        print(f"  avg amount/txn across surviving pairs -- "
              f"p50 {cc['avg_amt_pair'].median():,.0f}, "
              f"p{int(REINSURANCE_AVG_AMT_PCTL*100)} cut {reins_cut:,.0f}")
        _before = len(cc)
        cc["reinsurance_like"] = cc["avg_amt_pair"] > reins_cut
        cc = cc.loc[~cc["reinsurance_like"]]
        log_gate("cc: not reinsurance-shaped", len(cc), _before - len(cc),
                 "few-and-large treaty flows removed")

    cc["pair_type"] = "carrier_carrier"
    cc = cc.rename(columns={"a": "node_a", "b": "node_b"})
else:
    cc = pd.DataFrame(columns=["node_a", "node_b", "pair_type"])
    log_gate("cc: persistence", 0, 0, "no reciprocal pairs")

print(f"\nENSURED carrier<->carrier pairs: {len(cc):,}")

## 7. GATE — carrier ↔ law-firm ensured pairs

Direction is the signal here, and it runs opposite to the carrier↔carrier
logic. Carrier↔carrier subrogation looks like *balanced reciprocity*;
carrier↔law-firm does not:

- **firm → insurer** — plaintiff/recovery firm trust account remitting
  settlement proceeds against a WC lien or subrogation interest. The signal.
- **insurer → firm** — panel defense counsel. Standing panels are paid every
  month, so these are *maximally* persistent. Ranking on persistence alone
  surfaces defense panels and buries the actual recoveries — this gate
  exists specifically to drop them.
- **bidirectional** — recovery counsel: carrier funds pursuit, firm remits
  net of contingency fee. Kept.

In [ ]:
il = edges.loc[((edges["source_class"] == "insurer") & (edges["dest_class"] == "law_firm")) |
               ((edges["source_class"] == "law_firm") & (edges["dest_class"] == "insurer"))].copy()

if len(il):
    il["insurer_id"] = np.where(il["source_class"] == "insurer", il["source"], il["dest"])
    il["firm_id"] = np.where(il["source_class"] == "law_firm", il["source"], il["dest"])
    il["direction"] = np.where(il["source_class"] == "insurer", "i2f", "f2i")

    _keep = ["insurer_id", "firm_id", "amount_total", "volume_total",
             "n_months_active", "max_streak"]
    f2i = (il.loc[il["direction"] == "f2i", _keep]
           .rename(columns={c: f"{c}_f2i" for c in _keep[2:]}))
    i2f = (il.loc[il["direction"] == "i2f", _keep]
           .rename(columns={c: f"{c}_i2f" for c in _keep[2:]}))
    lf = f2i.merge(i2f, on=["insurer_id", "firm_id"], how="outer")
    for side in ("f2i", "i2f"):
        for c in _keep[2:]:
            lf[f"{c}_{side}"] = lf[f"{c}_{side}"].fillna(0)

    lf["amount_both"] = lf["amount_total_f2i"] + lf["amount_total_i2f"]
    lf["volume_both"] = lf["volume_total_f2i"] + lf["volume_total_i2f"]
    lf["recovery_ratio"] = lf["amount_total_f2i"] / lf["amount_both"].replace(0, np.nan)
    lf["months_max"] = lf[["n_months_active_f2i", "n_months_active_i2f"]].max(axis=1)
    lf["streak_max"] = lf[["max_streak_f2i", "max_streak_i2f"]].max(axis=1)
    lf["is_trust_firm"] = lf["firm_id"].map(trust_of).fillna(False).astype(bool)

    rr = lf["recovery_ratio"]
    lf["flow_pattern"] = np.select(
        [rr >= RECOVERY_RATIO_HIGH, rr <= RECOVERY_RATIO_LOW],
        ["FIRM_TO_INSURER", "INSURER_TO_FIRM"], default="BIDIRECTIONAL")

    print(f"Insurer<->law-firm pairs before gating: {len(lf):,}")
    print(lf["flow_pattern"].value_counts().to_string())

    _before = len(lf)
    _defense = lf.loc[lf["flow_pattern"] == "INSURER_TO_FIRM"].copy()
    lf = lf.loc[lf["flow_pattern"] != "INSURER_TO_FIRM"]
    log_gate("lf: recovery direction", len(lf), _before - len(lf),
             "defense-panel direction dropped")
    if len(_defense):
        _defense["excl_reason"] = "defense_panel_direction"
        print(f"  (dropped defense panels incl. "
              f"{int((_defense['months_max'] >= MIN_MONTHS).sum())} that were highly "
              f"persistent -- exactly the cases persistence-only scoring would rank top)")

    _before = len(lf)
    # Persistence, with ONE reasoned exemption. A hard consecutive-streak
    # requirement is biased against the very pattern this branch exists to
    # catch: subrogation recoveries are lumpy (settlements close irregularly
    # and arrive net of contingency fees), while an unbroken monthly run is
    # the signature of a retainer -- i.e. the DEFENSE panel we just dropped.
    # Trust accounts are admitted on months-active alone, because the
    # IOLTA/client-trust signal is independently high-precision and gapped
    # activity is expected behaviour for them rather than weak evidence.
    _months_ok = lf["months_max"] >= MIN_MONTHS
    _streak_ok = (lf["streak_max"] >= MIN_STREAK) | lf["is_trust_firm"]
    _exempted = int((_months_ok & lf["is_trust_firm"] & (lf["streak_max"] < MIN_STREAK)).sum())
    lf = lf.loc[_months_ok & _streak_ok]
    log_gate("lf: persistence", len(lf), _before - len(lf),
             f">= {MIN_MONTHS} months AND (>= {MIN_STREAK} streak OR trust account)")
    if _exempted:
        print(f"  ({_exempted} trust-account pair(s) admitted on months-active alone "
              f"-- lumpy remittance is expected, not weak evidence)")

    lf = lf.rename(columns={"insurer_id": "node_a", "firm_id": "node_b"})
    lf["pair_type"] = "carrier_lawfirm"
else:
    lf = pd.DataFrame(columns=["node_a", "node_b", "pair_type"])
    _defense = pd.DataFrame()
    print("No insurer<->law-firm edges found.")

print(f"\nENSURED carrier<->law-firm pairs: {len(lf):,}")

## 8. Clearinghouse labelling

A node reciprocally connected to a large share of admitted insurers is an
arbitration / settlement clearinghouse, not an ordinary counterparty. It is
the single strongest subrogation marker in the graph — so it is **labelled
and kept**, not silently dropped. It does get its drawn edges capped at
render time, since one node touching everything is what turns the picture
into a hairball.

In [ ]:
insurer_ids = set(nodes.loc[nodes["node_class"] == "insurer", "node_id"])
_pairs_all = pd.concat([cc[["node_a", "node_b"]], lf[["node_a", "node_b"]]], ignore_index=True) \
    if (len(cc) or len(lf)) else pd.DataFrame(columns=["node_a", "node_b"])

if len(_pairs_all):
    _deg = pd.concat([_pairs_all["node_a"], _pairs_all["node_b"]]).value_counts()
    _thresh = max(2, int(np.ceil(CLEARINGHOUSE_INSURER_FRACTION * len(insurer_ids))))
    clearinghouses = set(_deg[_deg >= _thresh].index)
    print(f"Clearinghouse threshold: reciprocally linked to >= {_thresh} "
          f"of {len(insurer_ids)} admitted insurers")
    if clearinghouses:
        for c in list(clearinghouses)[:10]:
            print(f"  HUB: {name_of.get(c, c)}  (degree {int(_deg[c])})")
    else:
        print("  (none at this threshold)")
else:
    clearinghouses = set()
    print("No ensured pairs -- skipping clearinghouse detection.")

nodes["is_clearinghouse"] = nodes["node_id"].isin(clearinghouses)

## 9. Node budget — best cases only

Score every surviving pair, sort best-first, and add pairs until the node
cap would be exceeded. Greedy rather than a global optimum: it guarantees
the cap, keeps the strongest evidence, and is trivial to explain when
someone asks why a given pair is or isn't on the chart.

Score components (all rank-normalised to [0,1] so no single term dominates
on raw scale, then weighted):

- persistence — months active / window
- streak — consecutive run / window
- evidence weight — log1p(total amount), rank-normalised
- directional clarity — carrier↔carrier: balance; carrier↔law-firm: recovery share
- trust-account bonus — the least ambiguous subrogation channel

In [ ]:
def rank_norm(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    if s.notna().sum() <= 1:
        return pd.Series(np.where(s.notna(), 1.0, 0.0), index=s.index)
    return s.rank(pct=True, na_option="bottom")


scored = []

if len(cc):
    c = cc.copy()
    c["persistence"] = c["months_min"] / N_WINDOW
    c["streak"] = c["streak_max"] / N_WINDOW
    c["evidence"] = rank_norm(np.log1p(c["amount_both"]))
    c["clarity"] = c["balance_ratio"]            # already 0..1
    c["trust_bonus"] = 0.0
    scored.append(c[["node_a", "node_b", "pair_type", "persistence", "streak",
                     "evidence", "clarity", "trust_bonus", "amount_both"]])

if len(lf):
    l = lf.copy()
    l["persistence"] = l["months_max"] / N_WINDOW
    l["streak"] = l["streak_max"] / N_WINDOW
    l["evidence"] = rank_norm(np.log1p(l["amount_both"]))
    # clarity: how unambiguously the money runs firm -> insurer
    l["clarity"] = l["recovery_ratio"].fillna(0.5)
    l["trust_bonus"] = np.where(l["is_trust_firm"], 1.0, 0.0)
    scored.append(l[["node_a", "node_b", "pair_type", "persistence", "streak",
                     "evidence", "clarity", "trust_bonus", "amount_both"]])

if scored:
    allp = pd.concat(scored, ignore_index=True)
    W = {"persistence": 0.30, "streak": 0.20, "evidence": 0.20,
         "clarity": 0.20, "trust_bonus": 0.10}
    allp["score"] = sum(allp[k] * w for k, w in W.items())
    allp = allp.sort_values("score", ascending=False).reset_index(drop=True)

    kept_rows, kept_nodes = [], set()
    for r in allp.itertuples(index=False):
        new = {r.node_a, r.node_b} - kept_nodes
        if len(kept_nodes) + len(new) > MAX_NODES:
            continue                                  # skip, keep scanning
        kept_nodes |= new
        kept_rows.append(r)

    final_pairs = pd.DataFrame(kept_rows)
    log_gate("budget: node cap", len(final_pairs), len(allp) - len(final_pairs),
             f"{len(kept_nodes)} nodes <= cap {MAX_NODES}")
else:
    allp = pd.DataFrame()
    final_pairs = pd.DataFrame(columns=["node_a", "node_b", "pair_type", "score"])
    kept_nodes = set()
    print("No pairs survived the gates -- loosen thresholds in section 0.")

print(f"\nFINAL: {len(final_pairs):,} pairs, {len(kept_nodes):,} nodes")
if len(final_pairs):
    _disp = final_pairs.copy()
    _disp["a"] = _disp["node_a"].map(name_of)
    _disp["b"] = _disp["node_b"].map(name_of)
    print(_disp[["a", "b", "pair_type", "persistence", "clarity", "score"]]
          .head(25).to_string(index=False))

## 10. Build and draw the ensured network

In [ ]:
import networkx as nx

PALETTE = {"insurer": "#2E5EAA", "law_firm": "#E08A1E", "other": "#B8BDC4"}
EDGE_COLORS = {"firm_to_insurer": "#2E8B57", "insurer_to_firm": "#C0504D",
               "insurer_to_insurer": "#4472C4", "other": "#CCCCCC"}


def edge_kind(uc: str, vc: str) -> str:
    if uc == "law_firm" and vc == "insurer":
        return "firm_to_insurer"
    if uc == "insurer" and vc == "law_firm":
        return "insurer_to_firm"
    if uc == "insurer" and vc == "insurer":
        return "insurer_to_insurer"
    return "other"


def build_network(final_pairs: pd.DataFrame, edges: pd.DataFrame) -> nx.DiGraph:
    """
    Build the network from the ENSURED PAIRS ONLY.

    Deliberately NOT the induced subgraph on the kept nodes: that would pull
    back in every edge between those nodes, including the ones the gates just
    rejected (failed persistence, unbalanced, reinsurance-shaped, defense
    direction). Re-adding them would quietly undo the whole funnel and put
    unvetted relationships on a chart labelled "ensured". Only the two
    directions of each surviving pair are drawn.
    """
    g = nx.DiGraph()
    if not len(final_pairs):
        return g

    wanted = set()
    for r in final_pairs.itertuples(index=False):
        wanted.add((r.node_a, r.node_b))
        wanted.add((r.node_b, r.node_a))          # pairs are undirected; keep both legs

    key = list(zip(edges["source"], edges["dest"]))
    sub = edges.loc[pd.Series(key, index=edges.index).isin(wanted)].copy()

    # cap edges drawn per clearinghouse so one hub can't swamp the layout
    hub_mask = sub["source"].isin(clearinghouses) | sub["dest"].isin(clearinghouses)
    non_hub, hub_e = sub.loc[~hub_mask], sub.loc[hub_mask]
    if len(hub_e) > MAX_HUB_EDGES_DRAWN:
        print(f"  capping clearinghouse edges: {len(hub_e):,} -> {MAX_HUB_EDGES_DRAWN} "
              f"(top by amount)")
        hub_e = hub_e.nlargest(MAX_HUB_EDGES_DRAWN, "amount_total")
    sub = pd.concat([non_hub, hub_e], ignore_index=True)

    for r in sub.itertuples(index=False):
        uc, vc = cls_of.get(r.source, "other"), cls_of.get(r.dest, "other")
        for nid, ncls in ((r.source, uc), (r.dest, vc)):
            if not g.has_node(nid):
                g.add_node(nid, node_class=ncls, name=str(name_of.get(nid, nid)),
                           trust=bool(trust_of.get(nid, False)),
                           hub=nid in clearinghouses)
        g.add_edge(r.source, r.dest, weight=float(r.amount_total),
                   volume=float(r.volume_total), months=int(r.n_months_active),
                   kind=edge_kind(uc, vc))
    return g


G = build_network(final_pairs, edges)
print(f"Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
_kinds = {}
for _, _, d in G.edges(data=True):
    _kinds[d["kind"]] = _kinds.get(d["kind"], 0) + 1
print("Edge kinds:", _kinds)

In [ ]:
def draw(g, title="Ensured subrogation network", figsize=(16, 11)):
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D

    if g.number_of_nodes() == 0:
        print("Empty network -- nothing to draw.")
        return None, None

    ins = [n for n, d in g.nodes(data=True) if d["node_class"] == "insurer"]
    firms = [n for n, d in g.nodes(data=True) if d["node_class"] == "law_firm"]

    # bipartite when law firms are present (recovery reads right-to-left),
    # spring otherwise
    if firms and ins:
        pos = {}
        for i, n in enumerate(sorted(ins, key=lambda x: g.degree(x), reverse=True)):
            pos[n] = (0.0, -i * (12 / max(len(ins), 1)))
        for i, n in enumerate(sorted(firms, key=lambda x: g.degree(x), reverse=True)):
            pos[n] = (7.0, -i * (12 / max(len(firms), 1)))
        for i, n in enumerate([x for x in g.nodes() if x not in pos]):
            pos[n] = (3.5, -i * 1.2 - 1)
    else:
        pos = nx.spring_layout(g, k=0.7, seed=42, iterations=80)

    fig, ax = plt.subplots(figsize=figsize)
    strength = {n: sum(d["weight"] for _, _, d in g.in_edges(n, data=True)) +
                   sum(d["weight"] for _, _, d in g.out_edges(n, data=True))
                for n in g.nodes()}
    smax = max(strength.values()) or 1.0
    sizes = [280 + 2000 * (math.log1p(strength[n]) / math.log1p(smax)) for n in g.nodes()]
    colors = [PALETTE[g.nodes[n]["node_class"]] for n in g.nodes()]
    borders = ["#B00020" if g.nodes[n]["trust"] else
               ("#6A0DAD" if g.nodes[n]["hub"] else "#FFFFFF") for n in g.nodes()]
    bw = [3.0 if (g.nodes[n]["trust"] or g.nodes[n]["hub"]) else 0.8 for n in g.nodes()]

    nx.draw_networkx_nodes(g, pos, node_size=sizes, node_color=colors,
                           edgecolors=borders, linewidths=bw, ax=ax)
    wmax = max((d["weight"] for _, _, d in g.edges(data=True)), default=1.0)
    for u, v, d in g.edges(data=True):
        nx.draw_networkx_edges(g, pos, edgelist=[(u, v)], ax=ax,
                               edge_color=EDGE_COLORS[d["kind"]],
                               width=0.6 + 3.5 * (math.log1p(d["weight"]) / math.log1p(wmax)),
                               alpha=0.7, arrows=True, arrowsize=12,
                               connectionstyle="arc3,rad=0.08")
    labels = {n: (str(g.nodes[n]["name"])[:26] + "…") if len(str(g.nodes[n]["name"])) > 26
              else str(g.nodes[n]["name"]) for n in g.nodes()}
    nx.draw_networkx_labels(g, pos, labels, font_size=7, ax=ax)

    ax.legend(handles=[
        Line2D([0], [0], marker="o", color="w", label="Insurer",
               markerfacecolor=PALETTE["insurer"], markersize=11),
        Line2D([0], [0], marker="o", color="w", label="Law firm",
               markerfacecolor=PALETTE["law_firm"], markersize=11),
        Line2D([0], [0], marker="o", color="w", label="Trust account (red ring)",
               markerfacecolor=PALETTE["law_firm"], markeredgecolor="#B00020",
               markeredgewidth=3, markersize=11),
        Line2D([0], [0], marker="o", color="w", label="Clearinghouse (purple ring)",
               markerfacecolor=PALETTE["insurer"], markeredgecolor="#6A0DAD",
               markeredgewidth=3, markersize=11),
        Line2D([0], [0], color=EDGE_COLORS["firm_to_insurer"], lw=3,
               label="firm → insurer (RECOVERY)"),
        Line2D([0], [0], color=EDGE_COLORS["insurer_to_firm"], lw=3,
               label="insurer → firm"),
        Line2D([0], [0], color=EDGE_COLORS["insurer_to_insurer"], lw=3,
               label="insurer ↔ insurer"),
    ], loc="upper left", fontsize=9, frameon=True)
    ax.set_title(f"{title}\n{g.number_of_nodes()} nodes · {g.number_of_edges()} edges · "
                 f"window {START_YM}..{END_YM} — node size ∝ log total amount", fontsize=12)
    ax.axis("off")
    plt.tight_layout()
    return fig, ax


fig, ax = draw(G)

## 10b. Interactive view (optional)

Single self-contained HTML — `cdn_resources="in_line"` embeds the JS so it
opens offline and travels as one file, which matters on an internal
network where CDNs are typically blocked.

In [ ]:
def draw_interactive(g, out_html):
    try:
        from pyvis.network import Network
    except ImportError:
        print("pyvis not installed -- skipping. pip install pyvis --break-system-packages")
        return None
    if g.number_of_nodes() == 0:
        print("Empty network -- nothing to render.")
        return None

    net = Network(height="800px", width="100%", directed=True, bgcolor="#FFFFFF",
                  font_color="#222222", notebook=False, cdn_resources="in_line")
    net.barnes_hut(gravity=-9000, central_gravity=0.25, spring_length=180)
    strength = {n: sum(d["weight"] for _, _, d in g.in_edges(n, data=True)) +
                   sum(d["weight"] for _, _, d in g.out_edges(n, data=True))
                for n in g.nodes()}
    smax = max(strength.values()) or 1.0
    for n, d in g.nodes(data=True):
        net.add_node(n, label=str(d["name"])[:30],
                     title=(f"{d['name']}\nclass: {d['node_class']}\n"
                            f"trust: {d['trust']}  clearinghouse: {d['hub']}\n"
                            f"total amount: {strength[n]:,.0f}"),
                     color={"background": PALETTE[d["node_class"]],
                            "border": "#B00020" if d["trust"] else
                                      ("#6A0DAD" if d["hub"] else "#666666")},
                     borderWidth=3 if (d["trust"] or d["hub"]) else 1,
                     size=12 + 28 * (math.log1p(strength[n]) / math.log1p(smax)))
    wmax = max((d["weight"] for _, _, d in g.edges(data=True)), default=1.0)
    for u, v, d in g.edges(data=True):
        net.add_edge(u, v, color=EDGE_COLORS[d["kind"]],
                     width=1 + 6 * (math.log1p(d["weight"]) / math.log1p(wmax)),
                     title=(f"{d['kind']}\namount: {d['weight']:,.0f}\n"
                            f"volume: {d['volume']:,.0f}\nmonths: {d['months']}"))
    net.write_html(str(out_html), notebook=False)
    print(f"Wrote {out_html}")
    return out_html


html_path = draw_interactive(G, OUT_DIR / "ensured_subrogation_network.html")

## 11. Persist results + run manifest

The run manifest records the thresholds used, the funnel counts, and the
final network shape — so any chart produced here can be traced back to the
exact configuration that produced it.

In [ ]:
tag = f"{START_YM}_{END_YM}"

if len(final_pairs):
    out = final_pairs.copy()
    out["node_a_name"] = out["node_a"].map(name_of)
    out["node_b_name"] = out["node_b"].map(name_of)
    out.to_csv(OUT_DIR / f"ensured_pairs_{tag}.csv", index=False)

    ent = nodes.loc[nodes["node_id"].isin(kept_nodes),
                    ["node_id", "name", "node_class", "naics_code", "naics_desc",
                     "has_trust_flag", "is_clearinghouse"]]
    ent.to_csv(OUT_DIR / f"ensured_entities_{tag}.csv", index=False)

if len(_defense):
    _d = _defense.copy()
    _d["insurer_name"] = _d["insurer_id"].map(name_of)
    _d["firm_name"] = _d["firm_id"].map(name_of)
    _d.to_csv(OUT_DIR / f"excluded_defense_panels_{tag}.csv", index=False)

pd.DataFrame(DROP_LOG).to_csv(OUT_DIR / f"gate_drop_log_{tag}.csv", index=False)

manifest = {
    "window": {"start": START_YM, "end": END_YM, "n_months": N_WINDOW,
               "snapshots_missing": len(missing)},
    "thresholds": {
        "strict_require_valid_naics": STRICT_REQUIRE_VALID_NAICS,
        "insurer_naics3": INSURER_NAICS3, "law_firm_naics4": LAW_FIRM_NAICS4,
        "min_months": MIN_MONTHS, "min_streak": MIN_STREAK,
        "min_balance_ratio": MIN_BALANCE_RATIO,
        "reinsurance_avg_amt_pctl": REINSURANCE_AVG_AMT_PCTL,
        "recovery_ratio_high": RECOVERY_RATIO_HIGH,
        "recovery_ratio_low": RECOVERY_RATIO_LOW,
        "clearinghouse_insurer_fraction": CLEARINGHOUSE_INSURER_FRACTION,
        "max_nodes": MAX_NODES,
    },
    "funnel": DROP_LOG,
    "entities": {"nodes_total": int(len(nodes)),
                 "insurers_admitted": int(n_ins), "law_firms_admitted": int(n_law),
                 "clearinghouses": sorted(str(name_of.get(c, c)) for c in clearinghouses)},
    "result": {"pairs": int(len(final_pairs)), "nodes": int(len(kept_nodes)),
               "graph_nodes": int(G.number_of_nodes()),
               "graph_edges": int(G.number_of_edges()),
               "within_budget": bool(len(kept_nodes) <= MAX_NODES)},
    "caveat": ("No transaction-level ground truth exists yet. 'Ensured' means "
               "highest structural confidence with ambiguous cases discarded, "
               "NOT verified subrogation. Precision is unmeasured until labels "
               "arrive from AF, claims integration, or payment addenda."),
}
with open(OUT_DIR / f"run_manifest_{tag}.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("\n=== FUNNEL ===")
print(pd.DataFrame(DROP_LOG).to_string(index=False) if DROP_LOG else "(no gates ran)")
print(f"\n=== RESULT ===")
print(f"  pairs: {len(final_pairs):,}   nodes: {len(kept_nodes):,}  "
      f"(budget {MAX_NODES}, within budget: {len(kept_nodes) <= MAX_NODES})")
print("\nWrote:")
for p in sorted(OUT_DIR.glob("*")):
    print(f"  {p}  ({p.stat().st_size:,} bytes)")